In [ ]:
#-------------------------------------------------------------------------------
# Name:        02_create_input.data
# Purpose:     Create missing polygon input data for modelling
#
# Author:      Gijs van den Dool
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1dgFGkQ0Orjt8SbPtpOvPNb1A9y4Odeng?usp=drive_link

# ISL buffers:
# This are the inital mask for the model, for which the lines are converted to
# polygons wide enough to capture cells on the perpendicular of the road.
# Suggested distance is: 20m, 3 cells at each side of the road. Wider buffers
# might pick up too much of the stable forests.

# ISL grid

In [ ]:
# check packages installed:
packages = !pip list

if "mapclassify" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install mapclassify -q

In [ ]:
import os

import geemap

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

from matplotlib import pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Local download of the labels prepared by the team and root of this notebook:
# change the root string to reflect your settings:

project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_ISL"
task_folder = "01_Input_Data"
file_name = "Iter4_02_ISL.geojson" # might be different for other areas

# Load the data
file_path = os.path.join(project_root, task_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File found")
else:
    print("File not found")

File found


In [ ]:
gdf_ISL_lines = gpd.read_file(file_path)

print(len(gdf_ISL_lines)) # total lines in the AoI
print(gdf_ISL_lines.crs)  # current projection
gdf_ISL_lines.head(3)

107
EPSG:4326


,width,color,note,min_date,max_date,url,image_date,geometry
0,1.0,"139,53,182,255",None,2022-03-30,2022-04-01,https://apps.sentinel-hub.com/eo-browser/?zoom...,2022-03-31,"MULTILINESTRING ((15.11883 2.27123, 15.11926 2..."
1,1.0,"139,53,182,255",None,2022-03-30,2022-04-01,https://apps.sentinel-hub.com/eo-browser/?zoom...,2022-03-31,"MULTILINESTRING ((15.12380 2.27214, 15.12434 2..."
2,1.0,"139,53,182,255",None,2022-03-30,2022-04-01,https://apps.sentinel-hub.com/eo-browser/?zoom...,2022-03-31,"MULTILINESTRING ((15.13005 2.27259, 15.13032 2..."


In [ ]:
def set_projection(gdf_input, org_CRS, new_CRS):
  # working on the projection system (CRS)
  if org_CRS != new_CRS:  # setting the new projection, buffers will be
                              # reprojected to the org_CRS
    gdf_output = gdf_input.to_crs(new_CRS)
  else:
    gdf_output = gdf_input

  print(gdf_output.crs)

  return gdf_output

In [ ]:
def create_buffer(road_line, buffer_distance, AoI_id):
#-------------------------------------------------------------------------------
# Name:        create_buffer
# Purpose:     Create a buffer polygon around all the roads in the label AoI
# Returns:     GeoPandas Dataframe
#-------------------------------------------------------------------------------
  #creating the buffer:
  road_buffer = road_line['geometry'].buffer(buffer_distance)
  road_buffer = gpd.GeoDataFrame(geometry=gpd.GeoSeries(road_buffer))
  road_buffer = road_buffer.rename(columns={0:'geometry'}).set_geometry('geometry')

  #polygon only:
  road_buffer  = road_buffer['geometry'].unary_union

  # converting the polygon to a geodataframe:
  road_buffer = gpd.GeoDataFrame(geometry=gpd.GeoSeries(road_buffer))
  road_buffer = road_buffer.rename(columns={0:'geometry'}).set_geometry('geometry')

  road_buffer["AoI_id"] = AoI_id

  return road_buffer

In [ ]:
buffer_distance = 30 # in meters
AoI_id = "ISL_02"   # will be different for other areas

# setting the projection
org_CRS = gdf_ISL_lines.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_ISL_lines = set_projection(gdf_ISL_lines, org_CRS, new_CRS)

# create the buffer:
gdf_ISL_buffer = create_buffer(gdf_ISL_lines, buffer_distance, AoI_id)
print(len(gdf_ISL_buffer))
gdf_ISL_buffer

EPSG:32632
1


,geometry,AoI_id
0,"MULTIPOLYGON (((1181684.937 252554.250, 118168...",ISL_02


In [ ]:
# reset the projection system and save the buffer:
gdf_ISL_buffer = gdf_ISL_buffer.set_crs(new_CRS) #
gdf_ISL_buffer = gdf_ISL_buffer.to_crs(org_CRS)

task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"buffer_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_ISL_buffer.to_file(file_path)

In [ ]:
# gdf_ISL_buffer.explore()

In [ ]:
#-------------------------------------------------------------------------------
# With the buffers created we can go to the next step: creating a grid based
# on the AoI

In [ ]:
def create_grid(AoI_polygon, grid_distance, AoI_id):
#-------------------------------------------------------------------------------
# Name:        create_grid
# Purpose:     Create a grid of polygons inside the AoI based on a grid distance,
#              the grid distance is based on the patch size, and preset to be
#              128x128px, or 1280x1280m
# Returns:     GeoPandas Dataframe
#-------------------------------------------------------------------------------
  minx, miny, maxx, maxy = AoI_polygon.total_bounds

  width = grid_distance
  height = grid_distance
  rows = int(np.ceil((maxy - miny) /  height))
  cols = int(np.ceil((maxx - minx) / width))
  leftOriginX = int(minx)
  rightOriginX = int(minx) + width
  topOriginY = int(maxy)
  bottomOriginY = int(maxy)- height

  polygons = []
  grid_id_list = []
  for i in range(cols):
    topY = topOriginY
    bottomY =bottomOriginY
    for j in range(rows):
      polygons.append(Polygon([(leftOriginX, topY), (rightOriginX, topY),
                              (rightOriginX, bottomY), (leftOriginX, bottomY)]))
      grid_id = f"GID_{AoI_id}_{i}_{j}"
      grid_id_list.append(grid_id)
      topY = topY - height
      bottomY = bottomY - height

    # end for
    leftOriginX = leftOriginX + width
    rightOriginX = rightOriginX + width
  # end for

  grid_polygon = gpd.GeoDataFrame({'geometry':polygons})
  grid_polygon["GID"]=grid_id_list


  return grid_polygon

In [ ]:
working_folder = "01_Input_Data"
file_name = "ISL_02.geojson" # might be different for other areas

# get grid file path
grid_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
os.path.isfile(grid_path)

True

In [ ]:
gdf_AoI_id = gpd.read_file(grid_path)

print(len(gdf_AoI_id)) # total lines in the AoI
print(gdf_AoI_id.crs
)  # current projection
gdf_AoI_id.head()
#gdf_AoI_id.explore()

1
EPSG:4326


,image_AoI,min_date,max_date,url,image_date,geometry
0,ISL_02,2022-03-30,2022-04-01,https://apps.sentinel-hub.com/eo-browser/?zoom...,2022-03-31,"POLYGON ((15.11400 2.26899, 15.18289 2.26927, ..."


In [ ]:
# setting the projection
org_CRS = gdf_AoI_id.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_AoI_id = set_projection(gdf_AoI_id, org_CRS, new_CRS)

EPSG:32632


In [ ]:
# create the grid:
# grid_distance = 1280 # 128px size at 10m ground resolution
grid_distance = 1280 # 256px size at 10m ground resolution

gdf_grid = create_grid(gdf_AoI_id, grid_distance, AoI_id)

# reset the projection:
new_CRS = "EPSG:32632"
gdf_grid = gdf_grid.set_crs(new_CRS)
gdf_grid.to_crs(org_CRS, inplace=True)

task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"grid_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_grid.to_file(file_path)

In [ ]:
gdf_grid.head(5)

,geometry,GID
0,"POLYGON ((15.11381 2.31583, 15.12525 2.31578, ...",GID_ISL_02_0_0
1,"POLYGON ((15.11376 2.30432, 15.12520 2.30427, ...",GID_ISL_02_0_1
2,"POLYGON ((15.11371 2.29280, 15.12516 2.29275, ...",GID_ISL_02_0_2
3,"POLYGON ((15.11366 2.28129, 15.12511 2.28124, ...",GID_ISL_02_0_3
4,"POLYGON ((15.11361 2.26978, 15.12506 2.26973, ...",GID_ISL_02_0_4


In [ ]:
print(gdf_grid.crs)
print(gdf_ISL_buffer.crs)

gdf_mask_full = gpd.overlay(gdf_ISL_buffer, gdf_grid, how='union')
gdf_mask_full.head()

EPSG:4326
EPSG:4326


,AoI_id,GID,geometry
0,ISL_02,GID_ISL_02_0_3,"POLYGON ((15.11921 2.27183, 15.11922 2.27184, ..."
1,ISL_02,GID_ISL_02_1_0,"POLYGON ((15.13593 2.30917, 15.13593 2.30918, ..."
2,ISL_02,GID_ISL_02_1_2,"MULTIPOLYGON (((15.13656 2.28529, 15.13653 2.2..."
3,ISL_02,GID_ISL_02_1_3,"MULTIPOLYGON (((15.12507 2.27232, 15.12509 2.2..."
4,ISL_02,GID_ISL_02_2_0,"MULTIPOLYGON (((15.14762 2.31145, 15.14764 2.3..."


In [ ]:
# gdf_mask_full.explore()

In [ ]:
# In gdf_mask_full both AoI_id	GID	are filled when there is a road buffer in the
# grid, when AoI_id is Null it means that there is not road buffer.
# To select the full GID with a road we first select all the cells which are not
# Null on "AoI_id", and then create a join on GID, selecting also the parts in the
# grid polygon which doesn't have a road buffer.

# xtrain = df.loc[df['Survive'].notnull(), ['Age','Fare', 'Group_Size','deck', 'Pclass', 'Title' ]]
# xtrain

# Select the road elements
xSelect = gdf_mask_full.loc[gdf_mask_full['AoI_id'].notnull(),["AoI_id","GID"]]
xSelect.head(5)


,AoI_id,GID
0,ISL_02,GID_ISL_02_0_3
1,ISL_02,GID_ISL_02_1_0
2,ISL_02,GID_ISL_02_1_2
3,ISL_02,GID_ISL_02_1_3
4,ISL_02,GID_ISL_02_2_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_mask_select= gdf_mask_full.merge(xSelect, on='GID', how='inner', suffixes=('_1', '_2'))

gdf_mask_select[['mask']] = gdf_mask_select[['AoI_id_1']] \
  .where(gdf_mask_select[['AoI_id_1']].isnull(), 1) \
  .fillna(0).astype(int)
gdf_mask_select.drop(['AoI_id_1'], axis=1, inplace=True)

gdf_mask_select = gdf_mask_select.loc[gdf_mask_select['GID'].notnull()]
print(len(gdf_mask_select))
gdf_mask_select.tail(10)


36


/var/folders/hf/1s_l6dt91218_yqmlwxj9pxm0000gn/T/ipykernel_83708/3110764797.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0).astype(int)


,GID,geometry,AoI_id_2,mask
27,GID_ISL_02_3_0,"POLYGON ((15.15959 2.31563, 15.15958 2.31229, ...",ISL_02,0
28,GID_ISL_02_3_1,"MULTIPOLYGON (((15.15954 2.30412, 15.15954 2.3...",ISL_02,0
29,GID_ISL_02_3_2,"MULTIPOLYGON (((15.15949 2.29261, 15.15944 2.2...",ISL_02,0
30,GID_ISL_02_3_3,"MULTIPOLYGON (((15.14950 2.28114, 15.14950 2.2...",ISL_02,0
31,GID_ISL_02_4_0,"POLYGON ((15.17104 2.31558, 15.17099 2.30407, ...",ISL_02,0
32,GID_ISL_02_4_1,"MULTIPOLYGON (((15.17099 2.30407, 15.17096 2.2...",ISL_02,0
33,GID_ISL_02_4_2,"MULTIPOLYGON (((15.16427 2.29259, 15.16427 2.2...",ISL_02,0
34,GID_ISL_02_4_3,"MULTIPOLYGON (((15.16548 2.28107, 15.16548 2.2...",ISL_02,0
35,GID_ISL_02_5_1,"POLYGON ((15.18243 2.30402, 15.18238 2.29251, ...",ISL_02,0
36,GID_ISL_02_5_2,"POLYGON ((15.18238 2.29251, 15.18233 2.28099, ...",ISL_02,0


In [ ]:
gdf_mask_select['mask'].value_counts()

mask
1    18
0    18
Name: count, dtype: int64

In [ ]:
working_folder = "02_WorkingFiles"

file_name = f"mask_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(project_root, working_folder, file_name)
gdf_mask_select.to_file(file_path)


In [ ]:
# The last step is to create the ImagePatch_AoI, and this is the unmodified
# grid polygon, which is the the GID where mask = 1

# selecting rows based on condition
rslt_df = gdf_mask_select.loc[gdf_mask_select['mask'] == 1, ["GID"]]
rslt_df.head(5)

,GID
0,GID_ISL_02_0_3
1,GID_ISL_02_1_0
2,GID_ISL_02_1_2
3,GID_ISL_02_1_3
4,GID_ISL_02_2_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_grid_select= gdf_grid.merge(rslt_df, on='GID', how='inner', suffixes=('_1', '_2'))
gdf_grid_select.head(5)

,geometry,GID
0,"POLYGON ((15.11366 2.28129, 15.12511 2.28124, ...",GID_ISL_02_0_3
1,"POLYGON ((15.12525 2.31578, 15.13670 2.31573, ...",GID_ISL_02_1_0
2,"POLYGON ((15.12516 2.29275, 15.13660 2.29271, ...",GID_ISL_02_1_2
3,"POLYGON ((15.12511 2.28124, 15.13655 2.28119, ...",GID_ISL_02_1_3
4,"POLYGON ((15.13670 2.31573, 15.14815 2.31568, ...",GID_ISL_02_2_0


In [ ]:
working_folder = "02_WorkingFiles"

file_name = f"ImagePatch_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(project_root, working_folder, file_name)

gdf_grid_select.to_file(file_path)